# 03 — ReAct from Scratch

**Model:** DeepSeek-V3 via Nebius AI Studio  
**Pattern:** ReAct (Reasoning + Acting)  
**Difficulty:** ⭐⭐⭐⭐☆

---

## The Problem

Most ReAct tutorials hand you `create_react_agent` and call it done. That works — until it doesn't, and you have no idea why the agent is stuck in a loop, calling the wrong tool, or ignoring its results.

This notebook builds ReAct **from first principles** — no prebuilt agents, no magic helpers. You'll see exactly:
- How the Thought → Action → Observation cycle is constructed
- How tool results are fed back into the prompt
- How to detect when the agent has reached a final answer
- Where loops come from and how to break them

## The ReAct Pattern

ReAct (Yao et al., 2023) interleaves reasoning traces with action execution:

```
Thought: I need to find the current price of gold.
Action: web_search("gold price today USD per ounce")
Observation: Gold is trading at $2,340 per troy ounce as of June 2025.
Thought: Now I have the price. I can answer the question.
Final Answer: Gold is currently $2,340 per troy ounce.
```

## Architecture

```
┌─────────────────────────────────────────────────────┐
│                  Custom ReAct Graph                  │
│                                                      │
│  ┌──────────┐    ┌────────────┐   ┌──────────────┐  │
│  │  Reason  │───▶│  Parse     │──▶│  Action or   │  │
│  │  Node    │    │  Output    │   │  Final Answer│  │
│  └────▲─────┘    └────────────┘   └──────┬───────┘  │
│       │                                  │          │
│       │   ┌──────────────────────────────┘ (action) │
│       │   ▼                                         │
│       │  ┌──────────┐                              │
│       └──│  Execute │  (observation fed back)      │
│          │  Tool    │                              │
│          └──────────┘                              │
└─────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
import os
import re
import math
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)

print("LLM ready.")

## Tool Registry

Instead of using LangChain's `@tool` decorator, we define tools as plain callables in a dictionary. This makes the dispatch logic transparent.

In [ ]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def web_search(query: str) -> str:
    results = tavily.search(query=query, max_results=2)
    snippets = [r["content"] for r in results.get("results", [])]
    return "\n".join(snippets) if snippets else "No results found."


def calculator(expression: str) -> str:
    safe_ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe_ns["__builtins__"] = {}
    try:
        return str(eval(expression, safe_ns))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"


# The agent will use tool names as strings — this registry maps names to callables
TOOL_REGISTRY = {
    "web_search": web_search,
    "calculator": calculator,
}

TOOL_DESCRIPTIONS = """
Available tools:
- web_search(query: str) -> Search the web for current facts, news, or data.
- calculator(expression: str) -> Evaluate a math expression. E.g. calculator("sqrt(144) + 2**8")
"""

print("Tools registered:", list(TOOL_REGISTRY.keys()))

## State Definition

We track the full reasoning trace as a list of strings. This is what gets fed back into the prompt at each step.

In [ ]:
class ReActState(TypedDict):
    question: str
    trace: List[str]          # Accumulated Thought/Action/Observation steps
    final_answer: Optional[str]
    step_count: int
    max_steps: int

## The ReAct Prompt

The system prompt defines the format the model must follow. This is the heart of the pattern — the model's output is structured text that we parse, not native tool calls.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a helpful research assistant that solves problems step by step.

{tool_descriptions}

You must follow this EXACT format for every response:

Thought: <your reasoning about what to do next>
Action: <tool_name>(<argument>)

OR if you have a final answer:

Thought: <your reasoning>
Final Answer: <your complete answer to the question>

Rules:
- Only use tools from the available list.
- One action per response.
- Always start with a Thought.
- Only output "Final Answer" when you are certain of the result.
""".format(tool_descriptions=TOOL_DESCRIPTIONS)

## Output Parser

We parse the model's text output to extract: Thought, Action (tool + args), or Final Answer.

In [ ]:
def parse_react_output(text: str) -> dict:
    """
    Parse a ReAct-format response.
    Returns: {"thought": str, "action": str|None, "action_input": str|None, "final_answer": str|None}
    """
    result = {"thought": "", "action": None, "action_input": None, "final_answer": None}

    thought_match = re.search(r"Thought:\s*(.+?)(?=\nAction:|\nFinal Answer:|$)", text, re.DOTALL)
    if thought_match:
        result["thought"] = thought_match.group(1).strip()

    final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    if final_match:
        result["final_answer"] = final_match.group(1).strip()
        return result

    # Parse Action: tool_name(argument)
    action_match = re.search(r"Action:\s*(\w+)\((.*)\)", text, re.DOTALL)
    if action_match:
        result["action"] = action_match.group(1).strip()
        result["action_input"] = action_match.group(2).strip().strip('"\'')

    return result

## Graph Nodes

In [ ]:
def reason_node(state: ReActState) -> dict:
    """Ask the LLM for the next Thought + Action (or Final Answer)."""
    # Build the full prompt from accumulated trace
    trace_text = "\n".join(state["trace"])
    user_prompt = f"Question: {state['question']}\n\n{trace_text}"

    response = llm.invoke([
        SystemMessage(content=REACT_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ])

    parsed = parse_react_output(response.content)
    print(f"\n[Step {state['step_count'] + 1}] Thought: {parsed['thought'][:100]}...")

    new_trace = state["trace"].copy()
    new_trace.append(f"Thought: {parsed['thought']}")

    if parsed["final_answer"]:
        new_trace.append(f"Final Answer: {parsed['final_answer']}")
        print(f"[Step {state['step_count'] + 1}] → Final Answer reached.")
        return {
            "trace": new_trace,
            "final_answer": parsed["final_answer"],
            "step_count": state["step_count"] + 1
        }

    if parsed["action"]:
        new_trace.append(f"Action: {parsed['action']}({parsed['action_input']})")
        print(f"[Step {state['step_count'] + 1}] → Action: {parsed['action']}")

    return {
        "trace": new_trace,
        "step_count": state["step_count"] + 1,
        "_parsed_action": parsed["action"],
        "_parsed_input": parsed["action_input"]
    }


def execute_tool_node(state: ReActState) -> dict:
    """Execute the tool the agent requested and record the Observation."""
    # Extract last action from trace
    last_action_line = [t for t in state["trace"] if t.startswith("Action:")][-1]
    match = re.search(r"Action:\s*(\w+)\((.*)\)", last_action_line)

    if not match:
        observation = "Could not parse action. Please try again."
    else:
        tool_name = match.group(1)
        tool_input = match.group(2).strip().strip('"\'')

        if tool_name not in TOOL_REGISTRY:
            observation = f"Tool '{tool_name}' does not exist. Available: {list(TOOL_REGISTRY.keys())}"
        else:
            print(f"   Executing {tool_name}({tool_input!r})...")
            observation = TOOL_REGISTRY[tool_name](tool_input)

    print(f"   Observation: {str(observation)[:120]}...")

    new_trace = state["trace"].copy()
    new_trace.append(f"Observation: {observation}")
    return {"trace": new_trace}


def should_continue(state: ReActState) -> str:
    if state.get("final_answer"):
        return "end"
    if state["step_count"] >= state["max_steps"]:
        print(f"\n[Router] Max steps ({state['max_steps']}) reached — stopping.")
        return "end"
    # Check if last trace entry is an action (needs tool execution)
    if state["trace"] and state["trace"][-1].startswith("Action:"):
        return "execute"
    return "reason"

## Build the Graph

In [ ]:
# We need to add _parsed_action and _parsed_input as optional fields
from typing import Optional

class ReActState(TypedDict, total=False):
    question: str
    trace: List[str]
    final_answer: Optional[str]
    step_count: int
    max_steps: int
    _parsed_action: Optional[str]
    _parsed_input: Optional[str]


builder = StateGraph(ReActState)
builder.add_node("reason", reason_node)
builder.add_node("execute", execute_tool_node)

builder.set_entry_point("reason")
builder.add_conditional_edges(
    "reason",
    should_continue,
    {"execute": "execute", "reason": "reason", "end": END}
)
builder.add_edge("execute", "reason")

graph = builder.compile()
print("ReAct graph compiled.")

## Live Demo

**Scenario:** Ask a question that requires multiple reasoning steps — first a search, then a calculation on top of the result.

In [ ]:
initial = {
    "question": (
        "What is the approximate land area of Egypt in square kilometers? "
        "Then calculate how many times larger that is than the area of Qatar (11,571 km²)."
    ),
    "trace": [],
    "final_answer": None,
    "step_count": 0,
    "max_steps": 8
}

result = graph.invoke(initial)

In [ ]:
print("\n" + "=" * 60)
print("FULL REASONING TRACE")
print("=" * 60)
for step in result["trace"]:
    print(step)
    print()

print("=" * 60)
print("FINAL ANSWER:", result.get("final_answer", "No answer produced."))

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Text-based tool dispatch** | Output parsed with regex — no native tool_calls needed |
| **Accumulated trace** | Full Thought/Action/Observation history in state |
| **Loop detection** | `max_steps` guard prevents infinite reasoning cycles |
| **Transparent internals** | You control parsing, routing, and dispatch — no magic |

## What's Next

**Notebook 04** upgrades single-agent reasoning to a two-agent system: a **Planner** that breaks goals into steps, and an **Executor** that carries them out. This separation dramatically improves performance on multi-step tasks.